# STEP 4-B — 최신 모델 비교 (2단계)

베이스라인으로 파이프라인이 살아있는 걸 확인했으니, 이제 좋은 모델을 찾습니다.

## 어느 단계를 비교하나

**2단계(병변 6종)를 기준으로 비교합니다.** 이유:

- 1단계(정상/이상)는 거의 5:5 이진 문제라 ResNet50 으로도 잘 됩니다.
  백본을 바꿔서 얻는 이득이 작습니다.
- 2단계가 어려운 쪽입니다 — 6종이 서로 닮았고 5.3배 불균형입니다.
  여기서 갈립니다.

마지막에 **가장 좋은 백본으로 1단계도 한 번 학습**해서 파이프라인 양쪽을 맞춥니다.

## 후보 (2026년 기준)

| 모델 | 계열 | 특징 |
|---|---|---|
| ResNet50 | 고전 CNN | 기준선 |
| EfficientNetV2-S | 효율 CNN | 가볍고 강함. 모바일 배포 1순위 |
| **ConvNeXt V2** | 현대 CNN | 질감 표현이 좋아 **피부에 잘 맞을 가능성** |
| Swin V2 | 계층적 ViT | 지역 패턴 + 전역 문맥 |
| EVA-02 | ViT (MIM) | 정확도 상한 확인용, 무거움 |
| SigLIP 2 | 이미지-텍스트 사전학습 | 소량 데이터에서 강한 편 |

> 💡 "가장 최신 = 가장 좋음"이 아닙니다. 데이터가 몇만 장 규모면
> 거대 모델은 오히려 과적합합니다. **실험으로 정합니다.**

## GPU 메모리 관리

Colab T4(16GB)에서 EVA-02 base 를 돌리려면 배치를 줄이고
`grad_accum` 으로 실효 배치를 키워야 합니다. 아래 코드가 자동 처리합니다.

📖 [`docs/basics/09_ViT와_최신_백본_지도_2026.md`](../docs/basics/09_ViT와_최신_백본_지도_2026.md)

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "imagehash", "pyarrow", "grad-cam"], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-20.3"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 0-b. 로컬에서 만든 데이터 불러오기

한국 PC 에서 `prepare_local.py` 로 전처리한 `dogskin_prepared.zip` 을 가져옵니다.

> 🚨 **AI Hub 는 해외 IP 다운로드를 차단**해서 Colab 에서는 원본을 받을 수 없습니다.
> 다운로드·전처리는 로컬에서, 학습만 여기서 합니다.
> → [`docs/cautions/06`](../docs/cautions/06_해외IP_다운로드_차단_우회.md)

**준비**: `dogskin_prepared.zip` 을 Google Drive 에 올려두세요 (Kaggle 이면 Add Input).

In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# Colab: Drive 의 zip 해제 / Kaggle: /kaggle/input 의 풀린 폴더에 링크
env.load_prepared()

# 다른 환경에서 학습했다면 체크포인트를 가져옵니다 (Colab → Kaggle 이주)
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 끊기면 체크포인트가 사라집니다. env.mount_drive() 확인.")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")

In [ ]:
import gc, json
import torch
from src import labels, split, crop, data, models, train, evaluate, stages
from src.config import MODEL_ZOO, MODEL_BY_KEY, CLASSES_STAGE1, NORMAL_LABEL

# ★ GPU 없이 진행하면 20~30배 느립니다. 없으면 여기서 멈춥니다.
#   (Colab 무료 한도를 넘기면 말없이 CPU 런타임을 줍니다 — 이걸 막습니다)
env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# 노트북 03 이 남긴 선택을 그대로 씁니다.
# ⚠️ 1단계와 2단계가 **다른 크롭**일 수 있습니다 — 03 의 감사에서 정상/병변 박스
#    배율 격차가 크면 1단계는 full 을 씁니다 (배율로 정답이 새는 걸 막기 위해).
_sel = env.work_root()/"stage1_threshold.json"
sel = json.loads(_sel.read_text()) if _sel.exists() else {}
BEST_CROP = sel.get("stage2_crop") or (
    (env.work_root()/"best_crop.txt").read_text().strip()
    if (env.work_root()/"best_crop.txt").exists() else "m1.5")
STAGE1_CROP = sel.get("stage1_crop", BEST_CROP)
print(f"2단계 크롭 {BEST_CROP} | 1단계 크롭 {STAGE1_CROP} | 사용 가능 {crop.available_tags()}")
if STAGE1_CROP != BEST_CROP:
    print("  → 03 의 감사 결과에 따라 1단계만 다른 크롭을 씁니다.")

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")

s1_all = stages.to_stage1(crop.switch_tag(df, STAGE1_CROP))   # 정상/이상 (전체)
s2_all = stages.to_stage2(crop.switch_tag(df, BEST_CROP))     # 병변 6종만
tr2, va2 = split.get_fold(s2_all, 0)   # ← 모델 비교는 이 분할로 고정
print(f"2단계  train {len(tr2):,} / val {len(va2):,}")

In [ ]:
models.available()

## 1. 2단계 — 모델별 학습

시간이 오래 걸립니다. 모델 6개면 GPU 몇 시간입니다.

### 세션이 끊겨도 괜찮습니다

체크포인트는 매 에폭 **Drive 로 백업**되고, `train.fit` 이
끝난 모델은 건너뛰고 끊긴 모델은 그 에폭부터 이어받습니다.
끊기면 **이 노트북을 위에서부터 그냥 다시 돌리세요.**

| 상황 | 다시 돌렸을 때 |
|---|---|
| 모델 3개까지 끝, 4번째에서 끊김 | 1~3 건너뜀 → 4번째를 이어받음 |
| 표까지 다 만든 뒤 끊김 | 전부 건너뛰고 평가만 다시 (몇 분) |
| 한 모델만 다시 학습하고 싶다 | `train.fit(..., resume=False)` |

> ⚠️ Drive 마운트를 안 했다면 이 보호가 없습니다 (0-b 셀이 경고합니다).
> 모델 하나당 약 600MB 를 씁니다. 6개면 3.6GB — Drive 여유를 확인하세요.
> 다 끝나고 `best.pt` 만 남기려면 `dogskin_work/checkpoints/*/last.pt` 를 지우세요.

In [ ]:
def run_stage2(spec, epochs=15, effective_batch=64):
    # 모델 하나로 2단계(병변 6종)를 학습하고 평가 결과를 돌려줍니다.
    #
    # ★ 건너뛰기/이어받기는 train.fit 이 알아서 합니다:
    #    끝난 모델은 건너뛰고, 세션이 끊겨 중간에 멈춘 모델은 그 다음 에폭부터.
    #    그래서 세션이 죽으면 이 셀을 그냥 다시 돌리면 됩니다.
    name = spec.key

    cfg = CFG(model_name=spec.timm_name, img_size=spec.img_size, epochs=epochs,
              balance_strategy="class_weight", monitor="macro_f1",
              exp_name=f"s2_{name}")
    bs = env.suggest_batch_size(spec.img_size, spec.scale)
    cfg = CFG(**{**cfg.to_dict(), "batch_size": bs,
                 "grad_accum": max(1, effective_batch // max(bs, 1))})

    m = models.build(spec, len(CLASSES), pretrained=True,
                     drop_rate=cfg.drop_rate, drop_path_rate=cfg.drop_path_rate,
                     img_size=spec.img_size)
    ltr, lva, dtr, _ = data.build_loaders(tr2, va2, cfg, model=m, classes=CLASSES)

    print(f"\n{'='*64}\n  {name}  |  {spec.img_size}px  |  "
          f"batch {bs} × accum {cfg.grad_accum} = 실효 {bs*cfg.grad_accum}\n{'='*64}")
    train.fit(m, ltr, lva, cfg, ds_train=dtr)   # ← 끝난 학습이면 즉시 반환

    # fit 이 best 가중치를 m 에 되돌려 놓았으므로 이 m 으로 바로 평가합니다
    _, lg, yy = train.evaluate_loader(m, lva, None, DEV, len(CLASSES), tta_hflip=True)
    rep = evaluate.full_report(lg, yy, CLASSES, show=False)
    print(f"  → {name}: macro-F1 {rep.metrics['macro_f1']:.4f} "
          f"(CI {rep.ci[1]:.4f}–{rep.ci[2]:.4f})")
    del m; gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
    return rep

In [ ]:
results = {}
for spec in MODEL_ZOO:
    try:
        results[spec.key] = run_stage2(spec)
    except torch.cuda.OutOfMemoryError:
        print(f"❌ {spec.key}: VRAM 부족 — img_size 를 줄이거나 건너뜁니다")
        gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f"❌ {spec.key}: {type(e).__name__}: {e}")

## 2. 결과 비교

In [ ]:
table = evaluate.compare_models(results)

### 표 읽는 법

- **macro_F1** 이 주 지표
- **CI_low ~ CI_high** 가 겹치는 모델끼리는 "더 낫다"고 말할 수 없습니다.
  0.78 vs 0.76 인데 CI 가 겹치면 그냥 우연입니다.
- **min_recall** (최악 클래스 재현율)이 낮으면 평균이 좋아도 위험합니다.
  A6(결절·종괴)를 계속 놓치는 모델은 못 씁니다 — 종양 감별이 필요한 병변입니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ks = list(results); f1 = [results[k].metrics["macro_f1"] for k in ks]
lo = [results[k].ci[1] for k in ks]; hi = [results[k].ci[2] for k in ks]
order = np.argsort(f1)

fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(ks))
ax.barh(y, [f1[i] for i in order], color="#4C78A8")
ax.errorbar([f1[i] for i in order], y,
            xerr=[[f1[i]-lo[i] for i in order],[hi[i]-f1[i] for i in order]],
            fmt="none", ecolor="black", capsize=3)
ax.set_yticks(y, [ks[i] for i in order]); ax.set_xlabel("2단계 macro-F1 (95% CI)")
ax.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()
print("💡 오차막대가 겹치면 성능 차이가 통계적으로 유의하지 않습니다.")

## 3. 앙상블

**서로 다른 계열**(CNN + ViT)을 섞을 때 효과가 큽니다. 틀리는 방식이 다르기 때문입니다.
같은 모델을 seed 만 바꿔 섞으면 이득이 적습니다.

In [ ]:
from src.models import Ensemble

TOP = list(table["model"].head(3))
print("앙상블 후보:", TOP)

# 다른 세션에서 학습했다면 로컬에 없습니다 — Drive 백업에서 되살립니다
member_models = []
for k in TOP:
    train.restore_from_persist(f"s2_{k}", verbose=False)
    member_models.append(
        models.load_checkpoint(str(env.work_root()/"checkpoints"/f"s2_{k}"/"best.pt"),
                               MODEL_BY_KEY[k], len(CLASSES)))
ens = Ensemble(member_models).eval()

In [ ]:
# ⚠️ 앙상블 멤버들의 입력 해상도/정규화가 다르면 로더를 따로 써야 합니다.
#    여기서는 첫 멤버 기준으로 통일합니다 (간단하지만 약간 손해).
cfg_e = CFG(img_size=MODEL_BY_KEY[TOP[0]].img_size, exp_name="s2_ensemble")
dl_e, _ = data.eval_loader(va2, cfg_e, model=member_models[0], classes=CLASSES)

_, lg, yy = train.evaluate_loader(ens, dl_e, None, DEV, len(CLASSES), tta_hflip=True)
results["ensemble"] = evaluate.full_report(lg, yy, CLASSES)
table = evaluate.compare_models(results)

## 4. 1단계도 최고 백본으로 학습

2단계 우승 백본으로 1단계(정상/이상)를 한 번 학습합니다.
이진 문제라 에폭이 적어도 됩니다.

> 앙상블이 1위여도 여기서는 **단일 모델 1위**를 씁니다 —
> 1단계는 모든 사진에 대해 항상 돌아가므로 추론 비용이 3배가 되면 부담이 큽니다.

In [ ]:
BEST_MODEL = next(k for k in table["model"] if k != "ensemble")
print("최고 단일 모델:", BEST_MODEL)
spec = MODEL_BY_KEY[BEST_MODEL]

tr1, va1 = split.get_fold(s1_all, 0)
cfg1 = CFG(model_name=spec.timm_name, img_size=spec.img_size, epochs=8,
           balance_strategy="none", exp_name=f"s1_{BEST_MODEL}")
bs = env.suggest_batch_size(spec.img_size, spec.scale)
cfg1 = CFG(**{**cfg1.to_dict(), "batch_size": bs,
              "grad_accum": max(1, 64 // max(bs, 1))})

m1 = models.build(spec, len(CLASSES_STAGE1), pretrained=True, img_size=spec.img_size)
dl_tr1, dl_va1, ds_tr1, _ = data.build_loaders(tr1, va1, cfg1, model=m1,
                                               classes=CLASSES_STAGE1)
train.print_status(cfg1.exp_name)
train.fit(m1, dl_tr1, dl_va1, cfg1, ds_train=ds_tr1)

In [ ]:
_, lg1, y1 = train.evaluate_loader(m1, dl_va1, None, DEV,
                                   len(CLASSES_STAGE1), tta_hflip=True)
bin1 = evaluate.binary_report(stages.stage1_scores(lg1), stages.binary_targets(y1),
                              target_recall=cfg1.target_recall_stage1)
THR1 = bin1["threshold"]

(env.work_root()/"best_model.json").write_text(json.dumps({
    "stage2": BEST_MODEL, "stage1": BEST_MODEL,
    "stage2_crop": BEST_CROP, "stage1_crop": STAGE1_CROP,
    "ensemble_members": TOP,
    "stage2_macro_f1": results[BEST_MODEL].metrics["macro_f1"],
    "stage1_auroc": bin1["auroc"], "stage1_threshold": THR1,
}, indent=2, ensure_ascii=False))
(env.work_root()/"stage1_threshold.json").write_text(json.dumps({
    "threshold": THR1, "target_recall": cfg1.target_recall_stage1,
    "auroc": bin1["auroc"], "precision_at_target": bin1["precision_at_target"],
    "stage2_crop": BEST_CROP, "stage1_crop": STAGE1_CROP, "model": BEST_MODEL,
}, indent=2, ensure_ascii=False))
print("저장: best_model.json, stage1_threshold.json")

## 5. 이어붙여서 다시 확인 ★

모델을 바꿨으니 **파이프라인 전체**를 다시 재야 합니다.
2단계 macro-F1 이 올라도 이어붙인 성능은 안 오를 수 있습니다 —
1단계가 병목이면 2단계 개선이 사용자에게 도달하지 않습니다.

In [ ]:
train.restore_from_persist(f"s2_{BEST_MODEL}", verbose=False)
m2 = models.load_checkpoint(str(env.work_root()/"checkpoints"/f"s2_{BEST_MODEL}"/"best.pt"),
                            spec, len(CLASSES))
cfg2 = CFG(model_name=spec.timm_name, img_size=spec.img_size)

va_all = split.get_fold(s1_all, 0)[1]                 # 정상 포함 전체 검증셋
# 각 모델에는 그 모델이 학습한 크롭을 먹입니다
va_s2 = crop.switch_tag(va_all, BEST_CROP, verbose=False) if STAGE1_CROP != BEST_CROP \
        else va_all
dl_p1, ds_p1 = data.eval_loader(va_all, cfg1, model=m1, classes=CLASSES_STAGE1)
dl_p2, ds_p2 = data.eval_loader(va_s2, cfg2, model=m2, classes=CLASSES)
assert len(ds_p1.df) == len(ds_p2.df)
assert (ds_p1.df["image_name"].to_numpy() == ds_p2.df["image_name"].to_numpy()).all()

_, lgp1, _ = train.evaluate_loader(m1, dl_p1, None, DEV, len(CLASSES_STAGE1), tta_hflip=True)
_, lgp2, _ = train.evaluate_loader(m2, dl_p2, None, DEV, len(CLASSES), tta_hflip=True)

pipe = stages.pipeline_report(stages.stage1_scores(lgp1), lgp2,
                              ds_p1.df["label_orig"].to_numpy(), threshold=THR1)
stages.plot_pipeline_confusion(pipe)

## 6. Mixup/CutMix 실험 (선택)

의료 이미지에서 Mixup 은 논쟁적입니다 — 존재하지 않는 병변 조합을 만들어내니까요.
실제로 도움이 되는지 직접 확인하세요.

In [ ]:
# cfg_mix = CFG(model_name=spec.timm_name, img_size=spec.img_size, epochs=15,
#               balance_strategy="class_weight",
#               mixup_alpha=0.2, cutmix_alpha=0.2, exp_name=f"s2_mixup_{BEST_MODEL}")
# m = models.build(spec, len(CLASSES), pretrained=True)
# ltr, lva, dtr, _ = data.build_loaders(tr2, va2, cfg_mix, model=m, classes=CLASSES)
# train.fit(m, ltr, lva, cfg_mix, ds_train=dtr)

## 7. 교차검증 (최종 성능 보고용)

fold 하나만 보면 운이 좋았을 수 있습니다. 5개 fold 평균 ± 표준편차가 정직한 숫자입니다.

⏱️ 시간이 5배 걸립니다. 최종 보고 직전에만 돌리세요.

In [ ]:
# fold_scores = []
# for k in range(CFG().n_folds):
#     t, v = split.get_fold(s2_all, k)
#     cfg_k = CFG(model_name=spec.timm_name, img_size=spec.img_size, epochs=15,
#                 use_fold=k, balance_strategy="class_weight",
#                 exp_name=f"cv_s2_{BEST_MODEL}_f{k}")
#     m = models.build(spec, len(CLASSES), pretrained=True, verbose=False)
#     ltr, lva, dtr, _ = data.build_loaders(t, v, cfg_k, model=m, classes=CLASSES)
#     r = train.fit(m, ltr, lva, cfg_k, ds_train=dtr, verbose=False)
#     fold_scores.append(r.best_score)
#     print(f"fold {k}: {r.best_score:.4f}")
#     del m; gc.collect(); torch.cuda.empty_cache()
# print(f"\n5-fold macro-F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

---
## ✅ 다음 단계

`05_평가_보정_GradCAM.ipynb`

거기서 **"이 모델을 실제로 써도 되는가"** 를 판단합니다.
정확도가 좋아도 배경을 보고 있으면 못 씁니다.

📖 함께 읽기:
- [`docs/basics/07_평가지표_의료AI_관점.md`](../docs/basics/07_평가지표_의료AI_관점.md)
- [`docs/basics/08_확률보정과_임계값_결정.md`](../docs/basics/08_확률보정과_임계값_결정.md)